# Class models — V vs continuation (soft fade size)

Not mid prediction. `mo_5s` is outcome only. See `LEAKAGE.md`.
Re-run: `python3 run_class_models.py` then refresh this notebook.


In [ ]:
from pathlib import Path
import json
from IPython.display import display, Markdown, Image

HERE = Path('.').resolve()
if not (HERE / 'out' / 'summary.json').exists():
    HERE = Path('applications/edge_lab/class_models').resolve()
summary = json.loads((HERE / 'out' / 'summary.json').read_text())
print('decision:', summary['sizing'][summary['classifiers']['primary_model']]['verdict']['decision'])
print('primary model:', summary['classifiers']['primary_model'])
log = summary['classifiers']['logistic']
print(f"logistic OOS AUC={log['auc_test']:.3f} Brier={log['brier_test']:.4f}")
gbm = summary['classifiers'].get('gbm') or {}
if gbm:
    print(f"GBM OOS AUC={gbm['auc_test']:.3f} Brier={gbm['brier_test']:.4f}")
sz = summary['sizing'][summary['classifiers']['primary_model']]
print('OOS soft primary:', sz['primary_soft_rule'])
print('vs hard:', sz['soft_rules'][sz['primary_soft_rule']]['lift_vs_hard_v_rule'])
print('vs always:', sz['soft_rules'][sz['primary_soft_rule']]['lift_vs_always_fade'])


In [ ]:
import pandas as pd
rows = []
for name, r in sz['soft_rules'].items():
    rows.append({
        'rule': name,
        'mean_net': r['pnl']['mean'],
        'lo': r['pnl']['lo'],
        'hi': r['pnl']['hi'],
        'd_hard': r['lift_vs_hard_v_rule']['delta'],
        'd_always': r['lift_vs_always_fade']['delta'],
        'clears_both': r['clears_hard'] and r['clears_always'],
    })
for name in ('always_fade', 'hard_v_rule'):
    b = sz['baselines'][name]
    rows.append({'rule': name, 'mean_net': b['pnl']['mean'], 'lo': b['pnl']['lo'], 'hi': b['pnl']['hi'],
                'd_hard': None, 'd_always': None, 'clears_both': None})
display(pd.DataFrame(rows))


In [ ]:
fig_dir = HERE / 'out' / 'figs'
for p in sorted(fig_dir.glob('*.png')):
    display(Markdown(f'### {p.name}'))
    display(Image(filename=str(p)))


## Leakage checklist
- [x] no `mo_5s` as feature
- [x] no `label` / `recovery_5s` as feature
- [x] chrono train/test
- [x] Promote gated on OOS soft-size lift CI, not AUC alone
- [x] no mid ML claim
